# Multi-Task Omics — `multibench` end-to-end tutorial

This notebook takes one dataset, **D11** (CITE-seq PBMC, 2,864 cells, RNA + ADT),
through the whole workflow: find a method, resolve its inputs, check its
parameters, run it, score it and plot the scores.

```
inputs_for  →  params_for  →  run  →  evaluate  →  plot
 (data)        (tuning)      (method)  (metrics)   (figure)
```

## 0. Setup

The first cell installs `multibench-sc` if it is missing. Methods run in their own
environments, downloaded only when you set `INSTALL_ENVS = True` (Linux or Colab).

<details>
<summary>Details</summary>

**Default.** With `INSTALL_ENVS = False` no environment is downloaded. Every cell
runs; section 4 loads a stand-in for the Matilda run.

**Running methods.** Set `INSTALL_ENVS = True` on Colab or a Linux machine to
download Matilda's prebuilt environment (no conda needed) and run it.

**On Colab**, choose a GPU runtime first (Runtime -> Change runtime type -> T4 GPU).
On a CPU runtime the smaller CPU build is installed and training is much slower.

**Stored tables.** Sections 7 and 8 read `results/` when the notebook runs from the
repository's `notebooks/` folder. Elsewhere, as on Colab, they read the package's
stored results, which have no `status` or run-time columns.

</details>

In [ ]:
import importlib.metadata, importlib.util, sys
if importlib.util.find_spec("multibench") is None:
    # keep the numpy / pandas this interpreter already has
    pins = [f"{p}=={importlib.metadata.version(p)}" for p in ("numpy", "pandas") if importlib.util.find_spec(p)]
    !{sys.executable} -m pip -q install "multibench-sc>=0.3" {" ".join(pins)}
    importlib.invalidate_caches()
    if importlib.util.find_spec("multibench") is None:        # not on PyPI: install from GitHub
        !{sys.executable} -m pip -q install "git+https://github.com/DSichang/scMultiBench.git" {" ".join(pins)}
else:
    print("multibench already installed")

In [ ]:
# False: no environment is downloaded; method cells use stand-in results.
# True (Linux or Colab): run the methods here; no conda needed.
# 1 env, 0.6 GB to download on a CPU host, 3.0 GB on a GPU host.
INSTALL_ENVS = False

In [ ]:
%matplotlib inline

import warnings
for _w in (FutureWarning, DeprecationWarning):   # hide deprecation noise
    warnings.filterwarnings("ignore", category=_w)

import pandas as pd
import multibench as mtb

print("multibench", mtb.__version__)

from pathlib import Path
# stored results used in sections 7 and 8
RESULTS = Path("results")

### Environments

Methods run in dedicated environments; the next cell lists the one each method uses.
Section 4 runs Matilda: the cell after it installs Matilda's environment when
`INSTALL_ENVS = True` on Linux.

<details>
<summary>Details</summary>

**From a terminal.** `multibench env install --methods Matilda --packed --run`
installs the same environment. Give `--methods` a comma-separated list, or use
`--category vertical` for every vertical method.

**`--packed`** downloads a prebuilt archive, so no conda is needed. Without it the
environment is built with conda from its lockfile.

**Dry run.** Without `--run`, `env install` only prints what it would install;
`multibench env doctor --methods Matilda` shows whether it is installed.

**Location.** Environments are unpacked under `mtb.config.DEFAULT.envs_dir`; set it
before the install cell to use another disk.

</details>

In [ ]:
DATASET  = "D11"
CATEGORY = "vertical"
MODALITIES = ["rna", "adt"]

METHODS = ["Concerto", "MOFA2", "Matilda", "Multigrate", "Seurat_WNN", "UINMF",
           "VIMCCA", "moETM", "scMDC", "scMM", "scMSI", "scMoMaT", "sciPENN", "totalVI"]

pd.DataFrame([{"method": m, "env": mtb.method_info(m)["env"]} for m in METHODS])

In [ ]:
import sys
if not INSTALL_ENVS:
    print("INSTALL_ENVS is False - no environment is downloaded")
elif sys.platform != "linux":
    print("method environments are linux-64 archives - skipped on", sys.platform)
else:
    plan = mtb.env.install(['Matilda'], category=CATEGORY)              # dry run: sizes only
    todo = [r for r in plan if not r["exists"]]
    print(f"{len(todo)} of {len(plan)} envs to download, {sum(r['archive_bytes'] or 0 for r in todo) / 1e9:.1f} GB")
    for r in mtb.env.install(['Matilda'], category=CATEGORY, packed=True, dry_run=False):
        print(f"{r['env']:20s} {r['state']}")

## 1. Discovery — which methods fit my data?

`find_methods` lists the methods that support a category; `method_info` describes one method.

In [ ]:
vertical_rna_adt = mtb.find_methods(category="vertical", modalities=["rna", "adt"])
print(len(vertical_rna_adt), "methods support vertical [rna, adt]")
print(sorted(vertical_rna_adt))

In [ ]:
info = mtb.method_info("Matilda")
{k: info[k] for k in ("id", "language", "categories", "tasks", "env", "status")}

## 2. Data — resolve the inputs

`mtb.data.fetch` downloads D11 once. `inputs_for` returns the files a method needs
from a dataset; `check=True` raises an error now if one is missing.

In [ ]:
mtb.data.fetch(DATASET)                                        # downloaded once, then reused
inputs = mtb.inputs_for(DATASET, CATEGORY, "Matilda",          # (dataset, category, method)
                       modalities=MODALITIES, check=True)
inputs

In [ ]:
# cell-type labels used later for the metrics
mtb.labels_for(DATASET)

## 3. Parameters — what can I tune?

`params_for` lists a method's parameters: `defaults` are passed on every run,
`tunable` are the options the method's script accepts. Change any of them with
`run(..., params={...})`.

<details>
<summary>Details</summary>

**Empty `tunable`.** 9 of these 14 methods expose no parameters on their command
line, so they cannot be tuned through the package. The second table shows which.

**Upstream defaults.** The values under `tunable` are the script's own defaults;
`p["effective"]` holds the value a run uses. The
[`params_for` reference](https://dsichang.github.io/scMultiBench/reference/discover/#multibench.discover.params_for) lists
the other keys.

</details>

In [ ]:
p = mtb.params_for("Matilda", CATEGORY, MODALITIES)
print("defaults:", p["defaults"])
print("tunable :", list(p["tunable"])[:10])

In [ ]:
rows = []
for m in METHODS:
    q = mtb.params_for(m, CATEGORY, MODALITIES)
    rows.append({"method": m, "n_tunable": len(q["tunable"]),
                 "defaults": q["defaults"],
                 "example_tunable": ", ".join(sorted(q["tunable"])[:4]) or "(none - hardcoded upstream)"})
pd.DataFrame(rows).sort_values("n_tunable", ascending=False).reset_index(drop=True)

## 4. Run a method

`run` runs Matilda in its environment and loads the embedding it writes.
`params={"epochs": 5}` shortens training for this demo. Without the environment,
the cell prints one line and loads a stand-in.

<details>
<summary>Details</summary>

**Stand-in.** The cell downloads the benchmark host's `run_all` outputs for `D11`
with `mtb.data.fetch_outputs`, reloads them with `mtb.load_batch` and reads
Matilda's embedding from them.

**Offline fallback.** If that download fails, `emb` is `None` and section 5 shows
Matilda's stored scores (`load_results(source="rerun")`) instead of computing them.

</details>

In [ ]:
import h5py

def stand_in(dataset, method):
    """`method`'s embedding from the benchmark host's run_all outputs; None if that download fails."""
    try:
        batch = mtb.load_batch(mtb.data.fetch_outputs(dataset), methods=[method])
        with h5py.File(Path(batch.out_dir) / f"{method}_{dataset}" / "embedding.h5") as f:
            emb = f["data"][()]
        print(f"stand-in: {method}'s embedding from the benchmark host's run_all outputs for {dataset} (fetch_outputs)")
        return emb
    except Exception as e:                                   # offline, or the outputs are not published yet
        print(f"stand-in: the package's stored scores for {method} ({type(e).__name__} from fetch_outputs: {e})")
        return None

check = mtb.scan(DATASET, CATEGORY, methods=["Matilda"])
if check.env_ok.any():
    res = mtb.run(method="Matilda", category=CATEGORY, inputs=inputs,
                  out_dir="/tmp/tutorial/Matilda_D11",
                  params={"epochs": 5})
    emb = res.output
else:
    print("no method environment on this host - the run is skipped; a stand-in computed elsewhere covers it")
    emb = stand_in(DATASET, "Matilda")

print("embedding:", None if emb is None else emb.shape)

### Not every method returns an embedding

Check a method's output kind before computing metrics: 12 of these methods return
an embedding, 2 return a graph. Scoring a graph as an embedding gives a
meaningless result and no error.

<details>
<summary>Details</summary>

**The two graph methods.**

- `Seurat_WNN` writes only a neighbour graph, so there is nothing to score.
- `scMoMaT` writes a KNN graph plus a UMAP embedding, which `run` returns in `res.extra`.

**With `run_all`.** `mtb.run_all` checks the kind for you. It scores scMoMaT through
its UMAP (status `CHAIN_OK_GRAPH_METHOD`) and reports Seurat_WNN as
`RUN_OK_NO_EMBEDDING`.

</details>

In [ ]:
from multibench.engine import registry

rows = []
for m in METHODS:
    v = registry.get(m).select(CATEGORY, set(MODALITIES))
    rows.append({"method": m, "kind": v.output.kind, "file": v.output.file,
                 "extra": ", ".join(f"{o.kind}:{o.file}" for o in v.extra_outputs) or "-"})
pd.DataFrame(rows).sort_values("kind").reset_index(drop=True)

## 5. Evaluate — scIB metrics

`evaluate` scores the embedding against the cell-type labels. Pass the labels in
the embedding's cell order.

<details>
<summary>Details</summary>

**Metrics.** `metrics="clustering"` computes ARI, NMI, ASW, iASW, iF1 and cLISI.

**Batch metrics.** D11 has a single batch, so batch metrics do not apply.

</details>

In [ ]:
import numpy as np

labels = pd.read_csv(mtb.labels_for(DATASET)["cty"])
labels = labels["x"].to_numpy() if "x" in labels.columns else labels.iloc[:, -1].to_numpy()

if emb is not None:
    scores = mtb.evaluate(emb, category=CATEGORY, metrics="clustering", labels=labels)
else:
    print("no embedding on this host for Matilda - its stored scores stand in")
    stored = mtb.load_results(CATEGORY, dataset=DATASET, source="rerun", methods=["Matilda"])
    scores = pd.DataFrame({"Value": dict(zip(stored.metric, stored.value))})
scores

## 6. Plot

`to_long` reshapes the scores into a long table; `plot.bubble` draws it.

In [ ]:
long = mtb.to_long(scores, method="Matilda", dataset=DATASET, category=CATEGORY)
fig = mtb.plot.bubble(long)
fig.set_dpi(110)
display(fig)

## 7. Comparing all 14 methods

Running all 14 methods takes hours, so this section loads their stored D11 results.
Seurat_WNN has no scores: it returns a graph, not an embedding.

In [ ]:
def stored_table(kind, dataset):
    """The `summary` or `long_all` table for `dataset`: from results/, else from the package's stored results."""
    if RESULTS.is_dir():
        return pd.read_csv(RESULTS / f"{kind}_{dataset}.csv")
    long = mtb.load_results(dataset=dataset, source="rerun")
    if kind == "long_all":
        return long
    return long.pivot_table(index="method", columns="metric", values="value").rename_axis(columns=None).reset_index()

if not RESULTS.is_dir():
    print("no results/ folder here - the package's stored results stand in (no status or run-time columns)")
summary = stored_table("summary", "D11")
summary[[c for c in ["method", "status", "run_sec", "ARI", "NMI", "ASW"]
         if c in summary.columns]]

In [ ]:
all_long = stored_table("long_all", "D11")
fig = mtb.plot.bubble(all_long)
fig.set_dpi(110)
display(fig)

---

## 8. The other integration categories

Each category has its own reference dataset. Repeat the calls above with its
`dataset`, `category` and a modality list the method accepts
(`mtb.method_info(m)["supports"]`). For a dataset with several label files, pass
`labels_for(dataset, category, method)` to `evaluate` (see Label order below).

| category | data | reference dataset |
|---|---|---|
| vertical | several modalities measured in the same cells | D11 (2,864 cells) |
| diagonal | RNA and ATAC measured in different cells | D28 (11,014 cells) |
| mosaic | several batches; only some share a modality | D45 (32,151 cells) |
| cross | several batches, each with all modalities | D52 (23,478 cells) |

<details>
<summary>Details: method counts</summary>

**Counts.** The next cell counts the methods with a variant for the dataset's files
(`scan`, `files_ok`); `find_methods` alone checks only the modality types. A
dataset must be on disk to be counted.

</details>

In [ ]:
SCENARIOS = {
    "vertical": dict(dataset="D11", modalities=["rna", "adt"]),
    "diagonal": dict(dataset="D28", modalities=["rna", "atac_gas"]),
    "mosaic":   dict(dataset="D45", modalities=["rna1", "rna2", "atac2", "atac3"]),
    "cross":    dict(dataset="D52", modalities=["rna1", "rna2", "adt1", "adt2"]),
}

for cat, s in SCENARIOS.items():
    ds = s["dataset"]
    if not (mtb.config.DEFAULT.data_path / ds).is_dir():
        print(f"{cat:9s} {ds:5s} -> not on disk; mtb.data.fetch({ds!r}) first to count its methods")
        continue
    got = mtb.find_methods(category=cat, modalities=[m.rstrip("123") for m in s["modalities"]])
    sc = mtb.scan(ds, cat, methods=got, verbose=False)      # keeps methods with a variant for this dataset's files
    print(f"{cat:9s} {ds:5s} -> {sc[sc.files_ok].method.nunique():2d} methods")

### Stored results for each category

<details>
<summary>Details</summary>

**Conos on D28.** Its ARI is near 0. `load_results(source="rerun")` flags this
row with `DegenerateRerunWarning`; leave it out when ranking.

</details>

In [ ]:
for cat, ds in [("diagonal", "D28"), ("mosaic", "D45"), ("cross", "D52")]:
    df = stored_table("summary", ds)
    cols = [c for c in ["method", "status", "run_sec", "ARI", "NMI", "ASW"] if c in df.columns]
    print(f"\n===== {cat}  ({ds}) =====")
    print(df[cols].to_string(index=False))

### Label order

A method that stacks several batches or modalities sets its own cell order.
`labels_for(dataset, category, method)` returns the label files in that order, and
`evaluate` takes the dict as is. A wrong order gives a wrong score and no error.

```python
mtb.evaluate(emb, labels=mtb.labels_for("D52", "cross", "StabMap"))   # cty3, cty1, cty2
```

<details>
<summary>Details</summary>

**In the stored results.** The `label_order` column of each stored summary shows the
order used. StabMap on D52 puts batch 3 first (`cty3+cty1+cty2`) and uniPort on D28
puts ATAC before RNA; `labels_for` with the method returns both orders.

**Concerto.** On D52 it also put batch 3 first, but
`labels_for("D52", "cross", "Concerto")` returns `cty1, cty2, cty3`. Score it with
`label_order=` set to the order in its `label_order` column.

**Your own dict.** A dict you build or reorder yourself goes in as is only in the
default order (`cty1, cty2, ...`; RNA before ATAC); name any other order with
`label_order=`.

**Fewer batches.** A method that uses only some label files takes only those keys:
UINMF on D52 uses `label_order=["cty1", "cty2"]`.

**With `run_all`.** `mtb.run_all` tries every label order that matches the cell
count, keeps the one with the highest ARI and reports it in the summary columns
`label_order` and `label_order_confidence`.

</details>

### Reading the bubble chart

In [`plot.bubble`](https://dsichang.github.io/scMultiBench/reference/plot/#multibench.plot.bubble.bubble), size is a rank
and colour a scaled value, both relative to the methods plotted, so small score
gaps can look large. Read the values in the table below the figure.

<details>
<summary>Details</summary>

**Example.** On D52, sciPENN (0.6947), UINMF (0.6905) and StabMap (0.6880) differ
by less than 0.007 ARI but get different bubble sizes.

</details>

In [ ]:
fig = mtb.plot.bubble(stored_table("long_all", "D52"))
fig.set_dpi(110)
display(fig)

stored_table("summary", "D52")[["method", "ARI", "NMI", "ASW"]]

### Run time

Some methods take hours even on small datasets. Check
`mtb.method_info(m)["runtime"]` before running a method.

<details>
<summary>Details</summary>

**Examples.**

| method | dataset (cells) | run time |
|---|---|---|
| sciPENN | D11 (2,864) | 143 s |
| scJoint | D28 (11,014) | 209 s |
| scMSI | D11 (2,864) | 2.3 h |
| MultiVI | D45 (32,151) | 3.3 h |
| sciCAN | D28 (11,014) | 4.4 h |

**Without a GPU**, training methods take much longer than these times.

**Time limits.** A `run_all(timeout=...)` run that exceeds the limit ends with
status `TIMEOUT`; allow hours for the slow methods.

</details>

## Summary

| Step | API |
|---|---|
| pick a method | `find_methods`, `method_info` |
| resolve data | `inputs_for`, `labels_for` |
| inspect tuning | `params_for` |
| run | `run(..., params={...})` |
| metrics | `evaluate`, `to_long` |
| figure | `plot.bubble` |
| environments | `mtb.env.*`, `multibench env doctor/install` |